In [5]:
# The code in this block comes directly from data_sort_and_split.ipynb
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

heart_data = pd.read_csv("heart.csv")

heart_data['Sex_F'] = heart_data['Sex'].map({'M': 0, 'F': 1})
heart_data['ExerciseAngina'] = heart_data['ExerciseAngina'].map({'N': 0, 'Y': 1})
heart_data = heart_data.drop(columns=['Sex'])

heart_data['ChestPainType'] = pd.Categorical(heart_data['ChestPainType'], categories=['ASY', 'ATA', 'NAP', 'TA'])
heart_data['RestingECG'] = pd.Categorical(heart_data['RestingECG'], categories=['Normal', 'LVH', 'ST'])
heart_data['ST_Slope'] = pd.Categorical(heart_data['ST_Slope'], categories=['Up', 'Flat', 'Down'])

categorical_cols = ['ChestPainType', 'RestingECG', 'ST_Slope']
heart_data = pd.get_dummies(heart_data, columns=categorical_cols, drop_first=True, dtype=int)

feature_matrix = heart_data.drop("HeartDisease", axis=1)
target_labels = heart_data["HeartDisease"]

features_train, features_test, targets_train, targets_test = train_test_split(
    feature_matrix,
    target_labels,
    test_size=0.20,
    random_state=42,
    stratify= target_labels
)

# Scaling performed to avoid features with larger ranges being seen as more significant to model
# Furthermore, it prevents data leakage by fitting to training data only.

scaler = StandardScaler()

# WWe 'fit' only on training data to prevent data leakage from the test set.
scaler.fit(features_train)

# We transform both sets using the training parameters so the scale is consistent.
features_train_scaled = scaler.transform(features_train)
features_test_scaled = scaler.transform(features_test)



In [6]:
# Single Hidden Layer Model Creation
import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense, InputLayer

# Setting a seed so our model returns the same results everytime
tf.random.set_seed(1)

#Actual model creation, but it has no layers yet
neural_network_model = Sequential()

#Input layer created with 15 neurons, one for every feature in our cleaned dataset
input_layer = InputLayer(shape=(15,))
neural_network_model.add(input_layer)

# Creating and adding hidden layer
# using the ((2/3(input layer size)) + 1 + (output layer size)) principle for number of neurons.
hidden_layer = Dense(12, activation= "relu")
neural_network_model.add(hidden_layer)

output_layer = Dense(1, activation='sigmoid')
neural_network_model.add(output_layer)

neural_network_model.compile(optimizer='adam', loss='binary_crossentropy',)

In [7]:
#Training the Neural Networks

# Defining Early Stopping to prevent overfitting on our 900-row dataset
# monitor='val_loss': We watch the error on the 20% validation set, not already seen data
# patience=5: If the error doesn't improve for 5 straight epochs, we stop.
# restore_best_weights=True: This rolls the model back to its peak performance moment.
early_stop = tf.keras.callbacks.EarlyStopping(
    monitor='val_loss',
    patience= 10,
    restore_best_weights=True
)

# Training the model
# We use a batch_size of 16 to give the Adam optimizer more updates per epoch.
# We set epochs to 80, but Early Stopping will likely stop it much sooner (the 30-50 range)
# We use a validation split here to monitor performance on unseen data
# DURING training, so we can stop before the model overfits
neural_network_model.fit(
    features_train_scaled,
    targets_train,
    epochs=80,
    batch_size=16, # The number of rows the model trains itself on before testing itself
    validation_split=0.2, #Like a practice quiz
    callbacks=[early_stop])

Epoch 1/80
37/37 ━━━━━━━━━━━━━━━━━━━━ 1s 7ms/step - loss: 0.6353 - val_loss: 0.5864
Epoch 2/80
37/37 ━━━━━━━━━━━━━━━━━━━━ 0s 4ms/step - loss: 0.5350 - val_loss: 0.5272
Epoch 3/80
37/37 ━━━━━━━━━━━━━━━━━━━━ 0s 3ms/step - loss: 0.4706 - val_loss: 0.4902
Epoch 4/80
37/37 ━━━━━━━━━━━━━━━━━━━━ 0s 3ms/step - loss: 0.4262 - val_loss: 0.4657
Epoch 5/80
37/37 ━━━━━━━━━━━━━━━━━━━━ 0s 4ms/step - loss: 0.3952 - val_loss: 0.4502
Epoch 6/80
37/37 ━━━━━━━━━━━━━━━━━━━━ 0s 4ms/step - loss: 0.3731 - val_loss: 0.4405
Epoch 7/80
37/37 ━━━━━━━━━━━━━━━━━━━━ 0s 3ms/step - loss: 0.3571 - val_loss: 0.4341
Epoch 8/80
37/37 ━━━━━━━━━━━━━━━━━━━━ 0s 3ms/step - loss: 0.3449 - val_loss: 0.4301
Epoch 9/80
37/37 ━━━━━━━━━━━━━━━━━━━━ 0s 3ms/step - loss: 0.3354 - val_loss: 0.4275
Epoch 10/80
37/37 ━━━━━━━━━━━━━━━━━━━━ 0s 4ms/step - loss: 0.3279 - val_loss: 0.4258
Epoch 11/80
37/37 ━━━━━━━━━━━━━━━━━━━━ 0s 4ms/step - loss: 0.3216 - val_loss: 0.4245
Epoch 12/80
37/37 ━━━━━━━━━━━━━━━━━━━━ 0s 4ms/step - loss: 0.3160 - val_lo

In [8]:
#Predicting with the models and turning probabilities into 1s and 0s

#Making the models predict based on the scaled testing data
model_probabilities = neural_network_model.predict(features_test_scaled)

#Converting to 1s and 0s
# Sigmoid outputs are between 0-1, but Precision/Recall/F1 need discrete categories.
model_predictions = (model_probabilities >= 0.5).astype(int)

6/6 ━━━━━━━━━━━━━━━━━━━━ 0s 9ms/step 


In [10]:
 #Retreiving Metrics for each mode, then comparing them
from sklearn.metrics import accuracy_score, f1_score, roc_auc_score, recall_score, precision_score

metrics_used = ["Accuracy", "Recall", "Precision", "f-1 Score", "ROC-AUC Score"]

model_metrics = []
model_metrics.append(accuracy_score(targets_test, model_predictions))
model_metrics.append(recall_score(targets_test, model_predictions))
model_metrics.append(precision_score(targets_test, model_predictions))
model_metrics.append(f1_score(targets_test, model_predictions))
model_metrics.append(roc_auc_score(targets_test, model_probabilities))



print(model_metrics)

[0.8858695652173914, 0.8823529411764706, 0.9090909090909091, 0.8955223880597015, 0.9335246293639408]


In [11]:
import numpy as np
import matplotlib.pyplot as plt

# 1. Setup the x-axis positions
# We use np.arange to create a list [0, 1, 2, 3, 4] for our 5 metrics
x = np.arange(len(metrics_used))
width = 0.35  # The width of each individual bar

plt.figure(figsize=(12, 7))

# 2. Create the grouped bars
# Why: x - width/2 shifts Model 1 to the left; x + width/2 shifts Model 2 to the right
rects1 = plt.bar(x - width/2, model1_metrics, width, label='Model 1 (1 Layer)', color='skyblue', edgecolor='black')

# 3. Add labels and aesthetics
plt.ylabel('Score (0.0 - 1.0)', fontsize=12)
plt.title('Heart Failure Prediction: Single Hidden Layer Model vs Two Hidden Layers Model', fontsize=14)
plt.xticks(x, metrics_used) # This replaces the [0,1,2,3,4] with your metric names
plt.legend()
plt.ylim(0, 1.1)
plt.grid(axis='y', linestyle='--', alpha=0.3)

#Add "Value Labels" on top of the bars
# Why: This makes your report easy to read so the viewer doesn't have to guess the exact score
plt.bar_label(rects1, padding=3, fmt='%.3f')

plt.tight_layout()
plt.show()

NameError: name 'model1_metrics' is not defined

<Figure size 1200x700 with 0 Axes>